## Imports

In [ ]:
import shutil
import numpy as np
import os
import json
import pandas as pd
from glob import glob
from astropy.io import fits
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import RandomFlip, RandomRotation

## Output directories

In [ ]:
FITS_DIR = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/FITS_BY_CHANNEL"
SOURCE_DIR = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/DATA_SOURCES/"

## Data sources

In [ ]:
FIRST_DIR = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F/FIRST"
LOFAR_DIR = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F/LoTSS"
NVSS_DIR = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F/NVSS"

CATALOG_FILE = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT.csv"

## Admin Functions

In [ ]:
def check_directory(output_folder, overwrite=True):
    if os.path.exists(output_folder):
        if overwrite:
            shutil.rmtree(output_folder)
            os.makedirs(output_folder)
            return True
        else:
            print(f"Folder {output_folder} already exists. Replace? (y/n)")
            choice = input().lower()

            if choice == 'y':
                shutil.rmtree(output_folder)
                os.makedirs(output_folder)
                return True
            else:
                return False

    else:
        os.makedirs(output_folder)
        return True
    
def count_folder_files(folder, extension=".fits"):
    if extension: 
        number = len([f for f in os.listdir(folder) if f.lower().endswith(extension)])
    else:
        number = len([f for f in os.listdir(folder)])

    print(f"Total FITS files in {folder}: {number}")
    # return number

## Functions

Adapted from ConstructData.py 

In [ ]:
def get_RADCAT_data(catalog_file=CATALOG_FILE, fits_folder=FITS_DIR):
    '''
    ARGS:
        catalog_file: path to the csv file containing the catalog data. It should have a "Type" column with the morphology type of each source, and the index should be the source names.
        fits_folder: path to the folder containing the fits files. The files should be named in the format "source_channel.fits", where channel is one of "FIRST", "LOFAR" or "NVSS". The folder should contain only the relevant fits files.
    RETURNS:
        data_images: a list of numpy arrays containing the image data for each channel. The shape of each array is (num_sources, width, height, num_channels). If the images have the same dimensions, then num_channels is 3 and the channels are in the last dimension. If the images have different dimensions, then num_channels is 1 and the channels are in separate arrays.
        labels: a numpy array containing the morphology type of each source.
        names: a numpy array containing the names of each source.
    '''
    same_dimensions = True 
    df = pd.read_csv(catalog_file, index_col=0)
    df = df[df["Type"].isin(["COMPACT", "FRI", "FRII"])]

    relevant_sources = sorted(df.index.to_list())
    relevant_files = os.listdir(fits_folder)

    names, labels = [],[]

    if same_dimensions:
        data_images = [np.zeros((len(relevant_sources), 128, 128, 3))] 
    else:
        data_images = [np.zeros((len(relevant_sources), 250, 250, 1)),  # FIRST
                       np.zeros((len(relevant_sources), 300, 300, 1)),  # LOFAR
                       np.zeros((len(relevant_sources), 30, 30, 1))]    # NVSS

    for i, source in enumerate(relevant_sources):

        names.append(source)
        labels.append(df.loc[source,"Type"])
        files = sorted([x for x in relevant_files if x.startswith(f"{source}_")])

        for j, file in enumerate(files):
            
            try:
                if same_dimensions:
                    data_images[0][i,:,:,j] = fits.getdata(fits_folder + "/" + file, memmap=False)
                else:
                    data_images[j][i,:,:,0] = fits.getdata(fits_folder + "/" + file, memmap=False)
            except Exception as e:
                print(f"Error for {source} in {file}")
                raise e

    return data_images, np.array(labels), np.array(names)


In [ ]:
def generate_augmented_data(Xs, y, names, save_dir, save_files):

    '''
    ARGS:
        Xs: a list of numpy arrays containing the image data for each channel. The shape of each array is (num_sources, width, height, num_channels). If the images have the same dimensions, then num_channels is 3 and the channels are in the last dimension. If the images have different dimensions, then num_channels is 1 and the channels are in separate arrays.
        y: a numpy array containing the morphology type of each source.
        names: a numpy array containing the names of each source.
        save_dir: the directory where the augmented data should be saved. The function will save two files in this directory: "augmented_images_no_clip.txt" and "augmented_images_clip.txt". Each file will contain the augmented images without and with sigma clipping respectively. The shape of the augmented images will be (num_augmented_images, width, height, num_channels). The function will also save a file "augmented_labels.txt" containing the labels of the augmented images, and a file "augmented_names.txt" containing the names of the augmented images.
        save_files: a boolean indicating whether the augmented data should be saved to disk. If False, the function will still generate the augmented data but will not save it to disk. This can be useful for testing the function without creating large files on disk.
    RETURNS:
        augmented_images_no_clip: a numpy array containing the augmented images without sigma clipping. The shape of the augmented images will be (num_augmented_images, width, height, num_channels).
        augmented_images_clip: a numpy array containing the augmented images with sigma clipping. The shape of the augmented images will be (num_augmented_images, width, height, num_channels).
        augmented_labels: a numpy array containing the labels of the augmented images.
        augmented_names: a numpy array containing the names of the augmented images.
    '''
    details = {}

    for morph in np.unique(y):
        num_of_sources = len(np.where(y == morph)[0])
        num_to_augment = 3000 - num_of_sources
        num_per_source = int(np.ceil(num_to_augment / num_of_sources))

        print(f"Augmenting {morph}: {num_of_sources}")
        print(f"\tNeed to augment at least {num_to_augment} images")
        print(f"\tAugmenting {num_per_source} images per source")
        print(f"\tThus, {num_per_source * num_of_sources} images will be added")

        details[morph] = num_per_source

    same_dimension = True if len(Xs) == 1 else False

    if os.path.exists(save_dir + "augmented_images.txt"):
        raise FileExistsError("Files already exist. Please remove them before running this function again.")

    with open(save_dir + "augmented_images.txt", "wb") as f_no_clip:
        for i in range(len(names)):
            print(f"Augmenting... {i+1}/{len(names)}", end='\r')

            aug_images = []
            aug_labels = []
            aug_names = []

            starting_seed = np.random.randint(2**32 - 1)
            for X in Xs:
                np.random.seed(starting_seed)
                aug_images.append([])
                min_value = np.min(X[i])
                for n in range(details[y[i]]):
                    seed = np.random.randint(2**32 - 1)
                    if same_dimension:
                        # (n, p, p, c)
                        aug_images[-1].append(
                            RandomFlip()(
                            RandomRotation(1, fill_mode="constant", fill_value=min_value)(X[i]) # 1 being -360 to 360 degrees
                            ))
                        aug_names.append(names[i])
                        aug_labels.append(y[i])

                    else:
                        # (n,) (p, p, 1)
                        aug_images[-1].append(
                            RandomFlip(seed=seed)(
                            RandomRotation(1,seed=seed, fill_mode="constant", fill_value=min_value)(X[i])) # 1 being -360 to 360 degrees
                            )
                        
                        aug_names.append(names[i])
                        aug_labels.append(y[i])
                        
                aug_images[-1] = np.array(aug_images[-1])
                
            # images = crop_normalize(aug_images)   
            [np.save(f_no_clip, X) if save_files else None for X in aug_images]
            # images = sigma_clipping(images, 3)
            # [np.save(f_clip, X) if save_files else None for X in images]

    if not save_files:
        # remove the files if they were not saved
        os.remove(save_dir + "augmented_images.txt")

    return aug_images, aug_labels, aug_names

In [ ]:
def check_empty_image_error(Xs, sources):
    for X in Xs:
        empty_indices = []
        len_channels = X.shape[-1]
        for i, image in enumerate(X):
            for channel in range(len_channels):
                if np.isnan(image[:,:,channel]).all():
                    empty_indices.append(i)
                    break
    
        if len(empty_indices) > 0:
            raise ValueError(f"Found {len(empty_indices)} empty images. These sources are : {sources[empty_indices]}")
    

def fill_partial_images(Xs, sources, save_dir, save_files):
    cut_off_sources = set()
    for X in Xs:
        for i in range(X.shape[0]):
            for channel in range(X.shape[-1]):
                im = X[i,:,:,channel]
                if np.isnan(im).any():
                    min_val = np.nanmin(im)
                    X[i,:,:,channel] = np.nan_to_num(im, nan=min_val)
                    cut_off_sources.add(sources[i])
        
    if len(cut_off_sources) > 0:
        print(f"Filled {len(cut_off_sources)} partial images. These sources are : {cut_off_sources}")
        if save_files:
            np.save(save_dir + "filled_partial_names.npy", list(cut_off_sources))

def print_distribution(dataset, dataset_name, total_images=None, spacing=20):
    total_images = len(dataset) if total_images is None else total_images
    print(f"\n--- {dataset_name} Distribution ---")
    unique, counts = np.unique(dataset, return_counts=True)
    print(f"{'Morphology':^{spacing}}|{'Count':^{spacing}}|{'Percentage':^{spacing}}")
    print(f"{'-'*spacing}|{'-'*spacing}|{'-'*spacing}")
    for i in range(len(unique)):
        print(f"{unique[i]:^{spacing}}|{counts[i]:^{spacing}}|{f'{counts[i]/len(dataset)*100:.2f}%':^{spacing}}")
    print(f"{'-'*spacing}|{'-'*spacing}|{'-'*spacing}")
    print(f"Total Images: {len(dataset)}/{total_images}\t{len(dataset)/total_images*100:.2f}%\n")

def add_fused_channels(X, fusion_specs):
    """
    Add multiple fused channels to a 3-channel image array.

    ARGS:
        X: numpy array of shape (num_sources, width, height, 3)
        fusion_specs: list of tuples like:
            [
                ("mean", None),
                ("max", None),
                ("weighted_mean", [0.5, 0.3, 0.2]),
            ]

    RETURNS:
        numpy array of shape (num_sources, width, height, 3 + len(fusion_specs))
    """
    if X.ndim != 4 or X.shape[-1] != 3:
        raise ValueError(f"Expected input shape (N, H, W, 3), got {X.shape}")

    fused_channels = [X]

    for method, weights in fusion_specs:
        if method == "mean":
            fused = np.mean(X, axis=-1, keepdims=True)
        elif method == "weighted_mean":
            if weights is None:
                raise ValueError("weights must be provided when method='weighted_mean'")
            weights = np.asarray(weights, dtype=X.dtype)
            if weights.shape != (3,):
                raise ValueError(f"weights must have shape (3,), got {weights.shape}")
            if np.isclose(np.sum(weights), 0):
                raise ValueError("Sum of weights must not be zero")
            weights = weights / np.sum(weights)
            fused = np.tensordot(X, weights, axes=([-1], [0]))[..., np.newaxis]
        elif method == "max":
            fused = np.max(X, axis=-1, keepdims=True)
        else:
            raise ValueError(f"Unsupported fusion method: {method}")

        fused_channels.append(fused.astype(X.dtype, copy=False))

    return np.concatenate(fused_channels, axis=-1)

## Code

In [ ]:
SAVE_FILES = True
pixel_type = "same_pixels/"
save_dir = SOURCE_DIR
use_fused_channel = True

check_directory(SOURCE_DIR)

In [ ]:
Xs, y, names = get_RADCAT_data()

In [ ]:
print(np.array(Xs).shape)
print(np.array(y).shape)
print(np.array(names).shape)

In [ ]:
if use_fused_channel:
    # print(f"Adding fused channel using method='{fusion_method}'")
    X_rgb = Xs[0][..., :3]   # always start from the original 3 channels
    Xs[0] = add_fused_channels(
        X_rgb,
        [
            ("mean", None),
            ("weighted_mean", [0.2, 0.6, 0.2]),
            # ("max", None),
        ]
    )
else:
    print("Skipping fused channel creation. Using original 3 channels.")

In [ ]:
check_empty_image_error(Xs, names)

In [ ]:
fill_partial_images(Xs, names, SOURCE_DIR, save_files=SAVE_FILES)

In [ ]:
train_val_index, test_index = train_test_split(range(len(y)), test_size=0.2, random_state=100, stratify=y)

In [ ]:
# X_test = crop_normalize([X[test_index] for X in Xs])
X_test = [X[test_index] for X in Xs]

In [ ]:
if SAVE_FILES:

    if pixel_type == "different_pixels/":
        np.save(SOURCE_DIR + "X_test_no_clip_FIRST.npy", X_test[0])
        np.save(SOURCE_DIR + "X_test_no_clip_LOFAR.npy", X_test[1])
        np.save(SOURCE_DIR + "X_test_no_clip_NVSS.npy", X_test[2])
        # X_test = sigma_clipping(X_test, 3)
        np.save(SOURCE_DIR + "X_test_clip_FIRST.npy", X_test[0])
        np.save(SOURCE_DIR + "X_test_clip_LOFAR.npy", X_test[1])
        np.save(SOURCE_DIR + "X_test_clip_NVSS.npy", X_test[2])
    else:
        np.save(SOURCE_DIR + "X_test.npy", X_test[0])
        # X_test = sigma_clipping(X_test, 3)
        # np.save(SOURCE_DIR + "X_test_clip.npy", X_test[0])
    
    np.save(SOURCE_DIR + "y_test.npy", y[test_index])
    np.save(SOURCE_DIR + "names_test.npy", names[test_index])

# del X_test


In [ ]:
Xs_train_val = [X[train_val_index] for X in Xs]
y_train_val = y[train_val_index]
names_train_val = names[train_val_index]

for key, val in {"TrainVal": y_train_val, "Testing": y[test_index]}.items():
    print_distribution(val, key, total_images=len(y))

# del Xs, y, names

In [ ]:
print(np.array(Xs_train_val).shape)
print(np.array(y_train_val).shape)
print(np.array(names_train_val).shape)

In [ ]:
X_aug, y_aug, names_aug = generate_augmented_data(Xs_train_val, y_train_val, names_train_val, save_dir=SOURCE_DIR, save_files=SAVE_FILES)

Augmenting COMPACT: 90
	Need to augment at least 2910 images
	Augmenting 33 images per source
	Thus, 2970 images will be added
Augmenting FRI: 1284
	Need to augment at least 1716 images
	Augmenting 2 images per source
	Thus, 2568 images will be added
Augmenting FRII: 634
	Need to augment at least 2366 images
	Augmenting 4 images per source
	Thus, 2536 images will be added


In [ ]:
print(np.array(X_aug).shape)
print(np.array(y_aug).shape)
print(np.array(names_aug).shape)

In [ ]:
if SAVE_FILES:
    np.save(SOURCE_DIR + "X_train_val.npy", Xs_train_val[0])    
    np.save(SOURCE_DIR + "y_train_val.npy", y_train_val)
    np.save(SOURCE_DIR + "names_train_val.npy", names_train_val)

    np.save(SOURCE_DIR + "X_aug.npy", X_aug[0])    
    np.save(SOURCE_DIR + "y_aug.npy", y_aug)
    np.save(SOURCE_DIR + "names_aug.npy", names_aug)

# del Xs_train_val, y_train_val, names_train_val

print("Finished!")

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, fixed

def view_image_channels(image_array, image_id):
    """Display all available channels for a given image ID"""
    channel_count = image_array.shape[-1]
    fig, axes = plt.subplots(1, channel_count, figsize=(4 * channel_count, 4))
    fig.suptitle(f'Image ID: {image_id}', fontsize=16)

    if channel_count == 1:
        axes = [axes]

    for channel in range(channel_count):
        axes[channel].imshow(image_array[image_id, :, :, channel], cmap='viridis')
        axes[channel].set_title(f'Channel {channel}')
        axes[channel].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
image_data = X_test[0]
max_images = image_data.shape[0] - 1

interact(
    view_image_channels,
    image_array=fixed(image_data),
    image_id=IntSlider(min=0, max=max_images, step=1, description='Image ID')
)


In [ ]:
image_data = Xs_train_val[0]
max_images = image_data.shape[0] - 1

interact(
    view_image_channels,
    image_array=fixed(image_data),
    image_id=IntSlider(min=0, max=max_images, step=1, description='Image ID')
)

# - FIRST: channel 0
# - LOFAR: channel 1
# - NVSS: channel 2
# - RADCAT: channel 3 

In [ ]:
image_data = X_aug[0]
max_images = image_data.shape[0] - 1

interact(
    view_image_channels,
    image_array=fixed(image_data),
    image_id=IntSlider(min=0, max=max_images, step=1, description='Image ID')
)